# Prognose ohne Erdschluss – Abschätzungen V0 bis V3

Ein **optionaler Teil** von groundfault: der Oberschwingungs-Fehlerstrom an einer Sammelschiene aus Netzdaten und – mit
einem Messgerät am Trafo – aus ungestörten Aufzeichnungen des Trafoknotens, **ohne Erdschlussversuch**. Vier Verfahren
nebeneinander:

| Verfahren | Daten | Ergebnis |
|---|---|---|
| **V0** klassische Abschätzung | U_m, I_C, Spule | oberhalb etwa der Resonanz der Sammelschiene eine obere Abschätzung, darunter nicht |
| **V1** Resonanzschranke | wie V0, eine Obergrenze G_max der Dämpfung | die Größenordnung in der Resonanz |
| **V2** Netzdaten | wie V0, Dämpfung d des Spulenreglers, Typschild, S″_k, C_1, eine angenommene Dämpfung G des MS-Netzes | eine Schätzung mit G und eine Hüllkurve über einen Bereich von G |
| **V3** Trafoknoten | wie V2, ungestörte Aufzeichnungen von HS-Spannungen, MS-Spannungen und Trafoströmen in einem Gerät | eine Schätzung mit gemessener Dämpfung des MS-Netzes |

Die Ordnungen des Nullsystems (3, 9, 15 …) haben eine Quelle, die erst im Fehler entsteht: Ohne Erdschluss gibt es für
sie keine Prognose. Mit Erdschlussversuchen oder aufgezeichneten Erdschlüssen `Netzmodell_Prognose.ipynb` verwenden
(V4 bis V7). Ausgangsfassung: `Vorlagen/Prognose_ohne_Erdschluss_DEFAULT.ipynb`. Verfahren: Online-Dokumentation,
*Forecast*.

1. **Konfiguration** – Netzdaten, Oberschwingungsspannungen, Annahmen, Aufzeichnungen des Trafoknotens, Grenzwerte.
2. **Netzdaten und Resonanz** – Vorimpedanz, Betriebskapazität, Resonanzordnung m_r und m/m_r.
3. **V3** – das MS-Netz aus jeder ungestörten Aufzeichnung mit ihren Prüfwerten und die zusammengefasste Schätzung der
   verlässlichen.
4. **V0 bis V3 nebeneinander** je Ordnung und Fehlerimpedanz, und I_H.
5. **Export** der Tabellen in den Prognoseordner des Arbeitsordners.

In [ ]:
%matplotlib inline
import datetime
import sys
from pathlib import Path

LANGUAGE = "de"                # Sprache von Tabellen, Hinweisen und Abbildungen: "de" | "en"
try:
    import groundfault as gf
except ImportError:            # nicht installiert: Quellordner eines Klons verwenden (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import forecast as fc
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, document, earthing,
                         equivalent_source, explorer, i18n, interactive, layout, legacy, pq, register, report, rms, sites,
                         transformer)

i18n.set_language(LANGUAGE)
# Arbeitsordner = dieser Ordner oder der erste Ordner darüber mit einem Kampagnenordner („01_Messkampagnen“
# oder „campaigns“); dessen Name gibt die Ordnerstruktur, ohne ihn gilt die Ordnerstruktur von LANGUAGE
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # Vorschau im Notebook = Schriftbild der exportierten Abbildungen


def show(df):
    """Tabelle mit Spaltennamen und Codes in der Ausgabesprache."""
    display(i18n.localize(df))


print("groundfault %s · Arbeitsordner: %s · Ordnerstruktur: %s · Schrift der Abbildungen: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Konfiguration

In [ ]:
NETWORK = "Demo-UW"             # Name des Netzes (Umspannwerk, Sammelschiene) – auch der Unterordner des Prognoseordners
DEMO    = True                  # Demo-Aufzeichnungen des ungestörten Trafoknotens anlegen, falls sie fehlen

# ---------------------------------------------------------------- Netzdaten (Demo-Werte)
U_N    = 20e3                   # Nennspannung des MS-Netzes in V
U_LE   = 20e3 / 3 ** 0.5        # Leiter-Erde-Spannung (Grundschwingung) in V
I_C    = 700.0                  # kapazitiver Erdschlussstrom in A (Spulenregler)
I_L    = 735.0                  # Spulenstrom in A (Spulenregler; None = I_C, abgestimmt)
D      = 0.03                   # Dämpfung I_w/I_C (Spulenregler): Reihenwiderstand d·X_C0 des Nullsystems in V2, V3
U_K    = 0.18                   # Kurzschlussspannung des Trafos (0.18 = 18 %)
S_RT   = 40e6                   # Bemessungsleistung des Trafos in VA
S_K    = 3.67e9                 # Kurzschlussleistung S″_k des HS-Netzes in VA
C_1    = 65e-6                  # Betriebskapazität des MS-Netzes je Leiter in F (Kabellängen); None = Q_C·C_E
Q_C    = None                   # C_1/C_E, wenn C_1 fehlt: Kabelnetze etwa 1, Mischnetze etwas mehr
R_X_UP = 0.03                   # R/X der Vorimpedanz (Trafo und HS-Netz) bei den Ordnungen – Ihre Annahme
R_X_HV = 0.1                    # R/X des HS-Netzes (korrigiertes Instrument von V3) – Ihre Annahme

# ---------------------------------------------------------------- Oberschwingungsspannungen und Fehlerimpedanzen
ORDERS   = (5, 7)               # Ordnungen des Mit- und Gegensystems (nicht 3, 9, 15 …: deren Quelle entsteht im Fehler)
VOLTAGES = {5: 250.0, 7: 130.0} # Demo-Werte: 95-%-Quantile in V (Leiter-Erde), z. B. einer Netzqualitätsmessung
Z_F      = (0.0, 2.5, 5.0)      # Fehlerimpedanzen in Ω
PQ_FILES    = None              # Export(e) eines PQ-Messgeräts an der Sammelschiene → VOLTAGES (Quantil × Faktor); relativ zum Arbeitsordner oder absolut
PQ_TIMEZONE = None              # Zeitzone der Geräteuhr, z. B. "Europe/Berlin" (None = Zeitstempel wie exportiert)
PQ_QUANTILE = 0.95              # Quantil der 10-min-Werte

# ---------------------------------------------------------------- Annahmen für V1 und V2 (Demo-Werte)
G_MAX   = 0.15                  # V1: Obergrenze der Dämpfung des MS-Netzes in S (None = V1 nicht gerechnet)
G       = 0.06                  # V2: angenommene Dämpfung in S (None = keine Schätzung V2)
G_RANGE = (0.02, 0.15)          # V2: Bereich der Dämpfung in S für die Hüllkurve (None = keine Hüllkurve)

# ---------------------------------------------------------------- V3: ungestörte Aufzeichnungen des Trafoknotens
RECORDINGS = "01_Messkampagnen/2026-03-10_Demo/1_Rohdaten/UW/TRU_*.cfg"   # Demo: Muster relativ zum Arbeitsordner oder eine Liste von .cfg-Pfaden
SIGN       = None               # Polarität der Trafoströme: +1 wie gemessen, −1 umgekehrt (None = aus allen Aufzeichnungen)
CT_ANGLE   = None               # Winkelfehler der Trafostromwandler in Grad: ein Wert oder {Ordnung: Winkel}
REFERENCE  = dict(recording="01_Messkampagnen/2026-03-10_Demo/1_Rohdaten/UW/TRU_02h_10_000kHz.cfg", channel="I1", reference="IRef")
# Referenzzange parallel zu einem Trafostrom (Demo: in der ersten Aufzeichnung) – gibt den Winkelfehler je Ordnung;
# None = CT_ANGLE

# ---------------------------------------------------------------- Grenzwerte – selbst festlegen
# groundfault enthält keine Grenzwerte. Nicht gesetzt (die Werte unten), listet das Notebook die Werte ohne Bewertung.
MIN_HV_SHARE  = None            # V3: kleinster Anteil der HS-getriebenen Schwankung einer verlässlichen Aufzeichnung (None = ohne Bewertung)
V0_RATIO      = None            # V0: unter m/m_r < V0_RATIO verdoppelt die Spalte V0_Regel_A den Wert von V0 (None = keine solche Spalte)
FUNDAMENTAL_A = None            # Grundschwingungsreststrom in A für I_RES (None = nur I_H)
LIMITS        = ()              # Grenzwerte des Reststroms in A, z. B. (I_1, I_2)
PQ_SAFETY_FACTOR = 1.0          # Faktor auf das Quantil der PQ-Messung (1.0 = keiner)
PQ_FACTORS    = None            # Faktor auf das Quantil nach Dauer der PQ-Messung {Mindesttage: Faktor} (None = PQ_SAFETY_FACTOR)

## 2 · Netzdaten und Resonanz

Aus Typschild und Kurzschlussleistung: X_T = u_k·U_n²/S_rT, X_Q = 1,1·U_n²/S″_k, die Vorimpedanz
Z_vor,m = m·(X_T + X_Q)·(R/X + j) und die Impedanz des HS-Netzes Z_HS,m = m·X_Q·(R/X + j) (für das korrigierte
Instrument von V3). Die Erdkapazität C_E = I_C/(3·ω₁·U_LE) und die Betriebskapazität C_1 ergeben die Parallelresonanz
der Sammelschiene, m_r = √(X_C1/X_vor) mit X_C1 = 1/(ω₁·C_1). Oberhalb etwa m_r sind Mit- und Gegensystem in der
Fehlerschleife kapazitiv, und die klassische Abschätzung V0 liegt über dem Strom; unterhalb m_r sind sie induktiv,
heben das kapazitive Nullsystem teilweise auf, und der Strom kann V0 um ein Mehrfaches übersteigen.

In [ ]:
X_T = fc.transformer_reactance(U_K, S_RT, U_N) if None not in (U_K, S_RT, U_N) else None
X_Q = fc.hv_network_reactance(S_K, U_N) if None not in (S_K, U_N) else None
X_UP = X_T + X_Q if None not in (X_T, X_Q) else None
C_E = fc.earth_capacitance(I_C, U_LE) if None not in (I_C, U_LE) else None
C1 = C_1 if C_1 is not None else (Q_C * C_E if None not in (Q_C, C_E) else None)
M_R = fc.resonance_order(X_UP, C1) if X_UP and C1 else None
Z_UP = {m: fc.at_order(X_UP, m, R_X_UP) for m in ORDERS} if None not in (X_UP, R_X_UP) else None
Z_HV = {m: fc.at_order(X_Q, m, R_X_HV) for m in ORDERS} if None not in (X_Q, R_X_HV) else None
NETWORK_DATA = pd.DataFrame([dict(X_T_Ohm=X_T, X_Q_Ohm=X_Q, X_up_Ohm=X_UP, C_E_uF=C_E * 1e6 if C_E else None,
                                  C_1_uF=C1 * 1e6 if C1 else None, m_r=M_R)])
show(NETWORK_DATA)
if M_R:
    show(pd.DataFrame([dict(order=m, f_Hz=50.0 * m, m_over_mr=m / M_R) for m in ORDERS]))
NOT_SET = [n for n, v in (("U_N", U_N), ("U_LE", U_LE), ("I_C", I_C), ("U_K", U_K), ("S_RT", S_RT), ("S_K", S_K),
                          ("R_X_UP", R_X_UP), ("R_X_HV", R_X_HV)) if v is None] + ([] if C1 else ["C_1 oder Q_C"])
if NOT_SET:
    print("ℹ Nicht gesetzt (Abschnitt 1): " + ", ".join(NOT_SET) + " – die Verfahren, die sie brauchen, entfallen.")

## 3 · V3 – das MS-Netz aus ungestörten Aufzeichnungen

Im ungestörten Betrieb schwanken die Oberschwingungsquellen des HS- und des MS-Netzes ständig. Eine vom HS-Netz
getriebene Schwankung ändert Sammelschienenspannung und Trafostrom um ΔU = Z_MS·ΔI, eine vom MS-Netz getriebene um
ΔU = −Z_vor·ΔI. Die HS-Spannung, korrigiert um den Spannungsfall über dem HS-Netz (E = U_HS + Z_HS·I_T), folgt nur den
HS-Quellen und dient als **Instrument**: Je 200-ms-Fenster ergeben die Abweichungen vom Mittel
Z_MS = ΣΔU·ΔE*/ΣΔI·ΔE* im natürlichen Drehsystem jeder Ordnung. Vorher wird der Winkelfehler der Trafostromwandler
korrigiert (Referenzzange `REFERENCE` oder `CT_ANGLE`).

Prüfwerte je Aufzeichnung: Korrelation zwischen Strom und Instrument (`corr_I_hv`), ein kapazitives Ergebnis mit
positivem Wirkanteil und der **Anteil der HS-getriebenen Schwankung** an der Varianz der Sammelschienenspannung
(`hv_share`). Der Anteil entscheidet: Treibt das MS-Netz den größten Teil der Schwankung, rückt die Schätzung zu
−Z_vor, und weder die Korrelation noch der Winkel zeigen das. Der Anteil ändert sich über den Tag (z. B. nachts höher),
deshalb zu mehreren Zeiten aufzeichnen. Die bestandenen Aufzeichnungen werden zusammengefasst – die Schätzung für die
Prognose.

In [ ]:
def path_in_work_folder(p):
    p = Path(p)
    return p if p.is_absolute() else WORK_FOLDER / p


if DEMO:                         # Demo: ein Messtag – sechs Aufzeichnungen von einer Minute, der HS-Anteil fällt zum Mittag
    folder = path_in_work_folder(Path(RECORDINGS).parent)
    folder.mkdir(parents=True, exist_ok=True)
    z_mv = {m: 1 / complex(g, m * 2 * np.pi * 50.0 * 65e-6) for m, g in ((5, 0.07), (7, 0.09))}   # die Wahrheit der Demo
    for k, (hour, hv, mv) in enumerate(((2, 0.10, 0.05), (6, 0.08, 0.07), (10, 0.04, 0.10), (12, 0.02, 0.13),
                                        (16, 0.05, 0.09), (22, 0.09, 0.06))):
        cfg = folder / ("TRU_%02dh_10_000kHz.cfg" % hour)
        if not cfg.exists():
            demo.make_undisturbed_transformer_recording(str(cfg), duration=60.0, Z_mv=z_mv, hv_fluctuation=hv,
                                                        mv_fluctuation=mv, ct_angle_deg={5: 2.0, 7: 2.8},
                                                        reference_clamp=(k == 0), seed=30 + k,
                                                        start=datetime.datetime(2026, 3, 10, hour, 0))
FILES = (sorted(WORK_FOLDER.glob(RECORDINGS)) if isinstance(RECORDINGS, str)
         else [path_in_work_folder(p) for p in RECORDINGS])
CT_TABLE = V3_TABLE = V3_POOLED = None
Z_MV = {}
CT = CT_ANGLE
if REFERENCE:
    CT_TABLE = transformer.ct_angle_from_reference(str(path_in_work_folder(REFERENCE["recording"])),
                                                   REFERENCE["channel"], REFERENCE["reference"], orders=ORDERS)
    display(Markdown("**Winkelfehler der Stromwandler** gegen die Referenzzange"))
    show(CT_TABLE)
    CT = CT_TABLE
if not FILES:
    print("ℹ V3 übersprungen – keine Aufzeichnungen (RECORDINGS, Abschnitt 1).")
elif Z_UP is None or Z_HV is None:
    print("ℹ V3 übersprungen – die Vorimpedanz braucht U_N, U_K, S_RT, S_K, R_X_UP und R_X_HV (Abschnitt 1).")
else:
    V3_TABLE, V3_POOLED = transformer.mv_network_series([str(f) for f in FILES], ORDERS, sign=SIGN, Z_hv=Z_HV,
                                                        Z_up=Z_UP, ct_angle_deg=CT, min_hv_share=MIN_HV_SHARE)
    cols = ["recording", "start", "order", "sequence", "Z_Ohm", "Z_angle_deg", "G_S", "B_S", "corr_I_hv", "hv_share",
            "reliable"]
    display(Markdown("**Je Aufzeichnung** (natürliches Drehsystem; Polarität der Ströme: %+g)" % V3_TABLE.attrs["sign"]))
    show(V3_TABLE[cols])
    for name, hints in V3_TABLE.attrs["hints"].items():
        for h in hints:
            print("ℹ %s: %s" % (name, h))
    display(Markdown("**Zusammengefasste Schätzung der verlässlichen Aufzeichnungen**"))
    show(V3_POOLED[["order", "sequence", "recordings", "Z_Ohm", "Z_angle_deg", "Z_low_Ohm", "Z_high_Ohm", "G_S", "B_S",
                    "hv_share", "names"]])
    Z_MV = {int(r.order): complex(r.R_Ohm, r.X_Ohm) for r in V3_POOLED.itertuples() if r.recordings}
    if MIN_HV_SHARE is None:
        print("ℹ Der Anteil der HS-getriebenen Schwankung (hv_share) ist gelistet, nicht bewertet – MIN_HV_SHARE ist nicht gesetzt (Abschnitt 1). Aufzeichnungen, in denen das MS-Netz den größten Teil der Schwankung treibt, werden dann mit zusammengefasst.")

## 4 · V0 bis V3 nebeneinander

Je Ordnung und Fehlerimpedanz: **V0** = 3·U_m/|Z_0,m + 3·Z_F| mit dem kapazitiven Z_0 aus I_C und der Spule;
**V1** = 3·U_m/|2/G_max + Z_0,m + 3·Z_F|; **V2** die volle Fehlerschleife 3·U_m/|Z_0 + Z_1 + Z_2 + 3·Z_F| mit
Z_1 = Z_2 = (1/Z_vor + G + j·m·ω₁·C_1)⁻¹ und Z_0 mit dem Reihenwiderstand d·X_C0 – die Schätzung mit `G` und die
Hüllkurve (größter Strom) über `G_RANGE`, dort ohne Dämpfung des Nullsystems; **V3** dieselbe Schleife mit dem
gemessenen MS-Netz (Abschnitt 3). Mit `V0_RATIO` verdoppelt die Spalte `V0_Regel_A` den Wert von V0 unterhalb
m/m_r < `V0_RATIO`. Die Zeilen `I_H` sind √Σ I_m² über die Ordnungen der Tabelle.

In [ ]:
COMPARISON = None
PQ = None
if PQ_FILES:                           # PQ-Messung der Sammelschiene: Quantil je Ordnung (größte Schleife/√3 bzw. größter Leiter) × Faktor
    files = [WORK_FOLDER / p for p in ([PQ_FILES] if isinstance(PQ_FILES, (str, Path)) else PQ_FILES)]
    PQ = pq.assess(pq.read(files, timezone=PQ_TIMEZONE), quantile=PQ_QUANTILE,
                   **(dict(factors=PQ_FACTORS) if PQ_FACTORS else dict(factor=PQ_SAFETY_FACTOR)))
    print(PQ.summary())
    for h in PQ.hints:
        print("ℹ", h)
    if PQ.valid:
        VOLTAGES = PQ.voltages() if PQ.reference == "LE" else {m: u / 3 ** 0.5 for m, u in PQ.voltages().items() if m % 3}
        print(i18n.format_text("VOLTAGES aus der PQ-Messung (Quantil × {:g}, Leiter-Erde bzw. Leiter-Leiter/√3): {}", PQ.factor,
                               ", ".join(i18n.format_text("{}: {:.1f} V", m, VOLTAGES[m]) for m in ORDERS if m in VOLTAGES)))
    else:
        print("⚠ Die PQ-Messung ist für die Bewertung nicht gültig (lückenlose Werte kürzer als jede Dauer von PQ_FACTORS) – keine Spannungen daraus; VOLTAGES aus Abschnitt 1 bleibt.")
if VOLTAGES is None:
    print("ℹ Keine Prognose – VOLTAGES ist nicht gesetzt (Abschnitt 1).")
elif None in (U_LE, I_C) or Z_UP is None:
    print("ℹ Keine Prognose – Netzdaten fehlen (Abschnitt 2).")
else:
    COMPARISON = fc.estimates_without_fault({m: VOLTAGES[m] for m in ORDERS if m in VOLTAGES}, I_C, U_LE, Z_UP,
                                            Z_F=tuple(Z_F), I_L=I_L, d=D or 0.0, C_1=C1, m_r=M_R, G_max=G_MAX,
                                            G=G, G_range=G_RANGE, Z_mv=Z_MV or None)
    if V0_RATIO is not None and M_R:
        rule = [v * (2.0 if isinstance(o, (int, np.integer)) and o / M_R < V0_RATIO else 1.0)
                for o, v in zip(COMPARISON.order, COMPARISON.V0_A)]
        COMPARISON.insert(list(COMPARISON.columns).index("V0_A") + 1, "V0_rule_A", rule)
        for zf in COMPARISON.Z_F_Ohm.unique():      # I_H der Regel: √Σ über die Ordnungen
            part = (COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order != "I_H")
            COMPARISON.loc[(COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order == "I_H"), "V0_rule_A"] = \
                float(np.sqrt((COMPARISON.loc[part, "V0_rule_A"] ** 2).sum()))
    show(COMPARISON)
    names = {"V0_A": "V0", "V0_rule_A": "V0 Regel", "V1_A": "V1", "V2_A": "V2",
             "V2_envelope_A": "V2 Hüllkurve", "V3_A": "V3"}
    methods = [c for c in names if c in COMPARISON.columns]
    for zf in COMPARISON.Z_F_Ohm.unique():
        h = COMPARISON[(COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order == "I_H")].iloc[0]
        text = ", ".join(i18n.format_text("{} {:.1f} A", names[c], h[c]) for c in methods)
        if FUNDAMENTAL_A is not None:
            text += " · I_RES: " + ", ".join(i18n.format_text("{} {:.1f} A", names[c], float(np.hypot(FUNDAMENTAL_A, h[c])))
                                             for c in methods)
        print(i18n.format_text("Z_F = {:g} Ω – I_H: ", zf) + text)
        if LIMITS and FUNDAMENTAL_A is not None:
            for limit in LIMITS:
                over = [names[c] for c in methods if np.hypot(FUNDAMENTAL_A, h[c]) > limit]
                print(i18n.format_text("   I_RES über {:g} A: ", limit) + (", ".join(over) or "–"))
    if FUNDAMENTAL_A is None:
        print("ℹ I_RES nicht gerechnet – FUNDAMENTAL_A ist nicht gesetzt (Abschnitt 1).")

## 5 · Tabellen exportieren

In [ ]:
OUT = Path(fc.forecast_folder(WORK_FOLDER, NETWORK))
TABLES = {"forecast_network_data": NETWORK_DATA, "forecast_ct_angle": CT_TABLE, "forecast_v3_recordings": V3_TABLE,
          "forecast_v3": V3_POOLED, "forecast_without_fault": COMPARISON}
TABLES = {k: v for k, v in TABLES.items() if v is not None and len(v) and v.notna().any().any()}
if TABLES:
    paths = report.export_tables(TABLES, str(OUT), prefix=report.slug(NETWORK) + "_without_fault_",
                                 formats=("csv", "md"))
    print("%d Tabellen-Dateien in %s" % (len(paths), OUT))
else:
    print("ℹ Nichts zu exportieren – keine Netzdaten, Aufzeichnungen oder Spannungen (Abschnitt 1).")